In [320]:
import torch
import numpy as np
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, CrossEntropyLoss
from torch.optim import Adam

In [321]:
dataset = load_wine()

In [322]:
x_treino, x_teste, y_treino, y_teste = train_test_split(dataset.data, dataset.target, test_size=0.2, random_state=42)

In [323]:
print(dataset.data.shape)

(178, 13)


In [324]:
print(dataset.target)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [325]:
model = Sequential(
    Linear(13,32),
    ReLU(),
    Linear(32,16),
    ReLU(),
    Linear(16,12)
)

In [326]:
print(model)

Sequential(
  (0): Linear(in_features=13, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=16, bias=True)
  (3): ReLU()
  (4): Linear(in_features=16, out_features=12, bias=True)
)


In [327]:
optimizer = Adam(model.parameters(), lr=0.001)
criterion = CrossEntropyLoss()
max_epochs = 500
batch = 8

In [328]:
total_bachs = len(x_treino)//batch
last_batch = len(x_treino) % batch > 0
has_last_b = last_batch>0
print(total_bachs)
print(last_batch)

17
True


In [329]:
def train_step_clas(x,y):
    optimizer.zero_grad()
    output = model(x)
    loss =  criterion(output, y)
    loss.backward()
    optimizer.step()

def train_model():
    for i_epochs in range(max_epochs):
        if has_last_b:
            batch_x = Tensor(x_treino[-last_batch:])
            batch_y = Tensor(y_treino[-last_batch:]).long()
            train_step_clas(batch_x, batch_y)
            

            for i_batchs in range(total_bachs):
                        batch_x = Tensor(x_treino[(i_batchs * batch):((i_batchs + 1) * batch)])
                        batch_y = Tensor(y_treino[(i_batchs * batch):((i_batchs + 1) * batch)]).long()
                        train_step_clas(batch_x, batch_y)

def eval_model():
    with torch.no_grad():
        x_teste_tensor = Tensor(x_teste)
        output = model(x_teste_tensor).cpu()
        result = torch.argmax(output, dim=1).cpu().numpy()
        print(f"ACC: {accuracy_score(y_teste, result)}")
        print(f"F1_SCORE: {f1_score(y_teste, result, average="macro")}")
        print(f"MATRIZ_CONFUSAO: \n{confusion_matrix(y_teste, result)}")
        

In [330]:
train_model()
eval_model()

ACC: 0.9722222222222222
F1_SCORE: 0.9761600681140911
MATRIZ_CONFUSAO: 
[[14  0  0]
 [ 1 13  0]
 [ 0  0  8]]
